In [23]:
import pandas as pd

In [24]:
return_average = pd.read_csv("bench_mark_return.csv", header = 0, index_col = 0)

In [25]:
return_average

,risk_parity,min_var,max_sharpe,paa,equal_strategy,equal_asset_weight
2019-01-30,0.055960,0.031233,-0.045906,-0.023335,0.004488,0.054332
2019-02-28,0.017790,0.007447,-0.022495,-0.016946,-0.003551,0.016301
2019-03-28,-0.006875,-0.012137,-0.013190,-0.012795,-0.011249,-0.004392
2019-04-26,0.022650,-0.002901,0.019038,0.021245,0.015008,0.024158
2019-05-24,-0.025741,-0.008459,-0.030936,-0.034561,-0.024924,-0.027220
...,...,...,...,...,...,...
2024-09-20,0.034817,0.049867,0.025878,0.001493,0.028014,0.034407
2024-10-18,0.003926,-0.001231,0.000756,0.016962,0.005103,0.002778
2024-11-15,-0.044229,-0.033668,-0.023999,-0.020194,-0.030522,-0.020486
2024-12-16,0.020330,0.032332,0.062063,0.078830,0.048389,0.036143


In [26]:
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.sandwich_covariance import cov_hac

def newey_west_tstat(returns, maxlags=1):
    """
    논문 방식에 따른 Newey-West t-통계량 계산 함수
    입력:
        returns: 수익률 벡터 (list, np.array, pd.Series)
        maxlags: Newey-West 보정에 사용할 최대 시차
    출력:
        (평균 수익률, NW 표준오차, NW t-통계량)
    """
    returns = np.asarray(returns)
    T = len(returns)
    X = np.ones((T, 1))  # 상수항만 포함 (평균 추정)
    
    model = sm.OLS(returns, X).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})
    nw_cov = cov_hac(model, nlags=maxlags)
    # nw_se = np.sqrt(nw_cov[0, 0])
    # t_stat = model.params[0] / nw_se
    
    return model.params[0], model.bse[0], model.tvalues[0]


In [27]:
from scipy.stats import t


In [28]:
return_average

,risk_parity,min_var,max_sharpe,paa,equal_strategy,equal_asset_weight
2019-01-30,0.055960,0.031233,-0.045906,-0.023335,0.004488,0.054332
2019-02-28,0.017790,0.007447,-0.022495,-0.016946,-0.003551,0.016301
2019-03-28,-0.006875,-0.012137,-0.013190,-0.012795,-0.011249,-0.004392
2019-04-26,0.022650,-0.002901,0.019038,0.021245,0.015008,0.024158
2019-05-24,-0.025741,-0.008459,-0.030936,-0.034561,-0.024924,-0.027220
...,...,...,...,...,...,...
2024-09-20,0.034817,0.049867,0.025878,0.001493,0.028014,0.034407
2024-10-18,0.003926,-0.001231,0.000756,0.016962,0.005103,0.002778
2024-11-15,-0.044229,-0.033668,-0.023999,-0.020194,-0.030522,-0.020486
2024-12-16,0.020330,0.032332,0.062063,0.078830,0.048389,0.036143


In [30]:
for col in return_average.columns:
    
    # 계산 실행
    mean_return, nw_se, nw_tstat = newey_west_tstat(return_average[col], maxlags=1)



    # 단측 검정 (우측): P(T > t)
    p_value = 1 - t.cdf(nw_tstat, df=len(return_average))
    print(f"Column: {col}")
    print(f"p-value = {p_value:.6f}")
    print(f"Mean Return = {mean_return:.6f}, NW SE = {nw_se:.6f}, NW t-stat = {nw_tstat:.6f}")

    


Column: risk_parity
p-value = 0.103958
Mean Return = 0.006079, NW SE = 0.004786, NW t-stat = 1.270124
Column: min_var
p-value = 0.015543
Mean Return = 0.008146, NW SE = 0.003708, NW t-stat = 2.196760
Column: max_sharpe
p-value = 0.114290
Mean Return = 0.008445, NW SE = 0.006958, NW t-stat = 1.213809
Column: paa
p-value = 0.297177
Mean Return = 0.003935, NW SE = 0.007358, NW t-stat = 0.534792
Column: equal_strategy
p-value = 0.084092
Mean Return = 0.006651, NW SE = 0.004781, NW t-stat = 1.391333
Column: equal_asset_weight
p-value = 0.100632
Mean Return = 0.006390, NW SE = 0.004957, NW t-stat = 1.289125


In [31]:
nw_tstat

np.float64(1.2891250406400871)

In [25]:
return_average

,date,return
0,2019-01-30,0.001730
1,2019-02-28,0.027784
2,2019-03-28,-0.006986
3,2019-04-26,0.019037
4,2019-05-24,-0.033522
...,...,...
71,2024-09-20,0.009221
72,2024-10-18,0.022761
73,2024-11-15,-0.013343
74,2024-12-16,0.106342


In [26]:
np.quantile(return_average['return'], 0.05)

np.float64(-0.05455205575)